<a href="https://colab.research.google.com/github/pradh/tools/blob/ml/dc-embed/BUILD_StatVarEmbeddings_CuratedDemographics300.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Prepare data by accessing curated CSV from drive


In [25]:
import pandas as pd

from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
creds, _ = default()

gc = gspread.authorize(creds)

sheet = gc.open_by_url("https://docs.google.com/spreadsheets/d/1GNI7iYHx2Q_RwJq-oqcqtiKbj8BJpyLghfkzUmnxS9k").worksheet("Main")

df = pd.DataFrame(sheet.get_all_records())
df

,Id,Owner,Name,Description Strings,Constraints
0,Count_Person,guha,Population,number of people; how many live; how many in; number of residents;,
1,Count_Person_PerArea,guha,Population Density,number of people per square mile; how crowded,
2,GrowthRate_Count_Person,guha,Population Growth Rate,Birth rate; Fertility rate; Natural increase rate; Rate of population increase; Population growth; Population change rate; annual population growth rate; rate of population change,
3,IncrementalCount_Person,guha,Population Change,change in population; change in number of people;,
4,LifeExpectancy_Person,guha,Life Expectancy,Average lifespan; Expected lifespan; Average length of life; Length of life on average; Duration of life; Average length of time spent living; death rate; mortality rate,
...,...,...,...,...,...
323,Count_Person_Male_BelowPovertyLevelInThePast12Months_NativeHawaiianOrOtherPacificIslanderAlone,shanth,"Population: Male, Below Poverty Level in The Past 12 Months, Native Hawaiian or Other Pacific Islander Alone",,gender : povertyStatus : race
324,Count_Person_Male_BelowPovertyLevelInThePast12Months_SomeOtherRaceAlone,shanth,"Population: Male, Below Poverty Level in The Past 12 Months, Some Other Race Alone",,gender : povertyStatus : race
325,Count_Person_Male_BelowPovertyLevelInThePast12Months_TwoOrMoreRaces,shanth,"Population: Male, Below Poverty Level in The Past 12 Months, Two or More Races",,gender : povertyStatus : race
326,Count_Person_Male_BelowPovertyLevelInThePast12Months_WhiteAlone,shanth,"Population: Male, Below Poverty Level in The Past 12 Months, White Alone",,gender : povertyStatus : race


In [26]:
TEXT2SV = {}

def add_sv(name, sv):
  if not name:
    return
  if name in TEXT2SV:
    TEXT2SV[name].append(sv)
  else:
    TEXT2SV[name] = [sv]

for _, row in df.iterrows():
  name = row['Name'].strip()
  sv = row['Id'].strip()
  add_sv(name, sv)
  for desc in row['Description Strings'].split(';'):
    add_sv(desc.strip(), sv)

texts = sorted(list(TEXT2SV.keys()))
dcids = [','.join(TEXT2SV[k]) for k in texts]

## 2. Build embeddings

Uses a pre-trained model to build embeddings for the above SV strings.

Must run this before you can search.  Downloading the model (~80MB) takes a minute maybe and embedding building is another minute or two.

TODO: try the larger / better(?) multi-QA model ([see this](https://www.sbert.net/docs/pretrained_models.html#semantic-search)).

In [ ]:
%%capture
!pip install -U sentence-transformers
!pip install datasets

In [27]:
from sentence_transformers import SentenceTransformer, util

# Download model
model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = model.encode(texts, show_progress_bar=True)

import pandas as pd
embeddings = pd.DataFrame(embeddings)
embeddings['dcid'] = dcids
embeddings

Batches:   0%|          | 0/15 [00:00<?, ?it/s]

,0,1,2,3,4,5,6,7,8,9,...,375,376,377,378,379,380,381,382,383,dcid
0,0.040603,0.023805,0.031632,0.021145,0.032872,0.016079,-0.067156,0.021720,-0.015401,-0.035370,...,0.031841,0.024510,0.040491,-0.034064,0.011583,-0.006351,-0.053033,-0.007523,0.003443,LifeExpectancy_Person
1,0.099664,0.013362,0.047402,0.056418,0.060111,-0.000517,-0.014039,0.017880,-0.044517,-0.044837,...,0.028126,0.061166,0.034343,-0.080750,-0.022892,-0.009679,-0.051830,-0.085939,-0.000914,LifeExpectancy_Person
2,0.027830,-0.006854,0.002610,0.036188,0.004634,-0.011708,-0.036925,0.091316,-0.057010,-0.038456,...,-0.002330,0.054654,0.057481,-0.062074,-0.018263,0.029759,-0.061553,-0.004894,0.008172,LifeExpectancy_Person
3,0.038167,-0.056210,-0.012850,0.034265,-0.016526,-0.010918,-0.027130,0.039239,-0.003930,0.040157,...,-0.003088,0.030649,-0.061457,0.014968,0.057762,0.005399,-0.075122,-0.007563,-0.002828,FertilityRate_Person_Female
4,0.011553,0.083289,-0.061666,0.007861,0.014624,0.007582,-0.071454,0.031597,-0.012532,0.038053,...,-0.056393,-0.016026,-0.017674,-0.027990,0.100812,0.005074,-0.029275,0.090665,-0.053679,"GrowthRate_Count_Person,FertilityRate_Person_Female"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
455,0.125208,-0.066474,0.034849,0.026156,-0.027884,0.092485,0.030373,0.082371,-0.031627,0.001114,...,-0.054567,0.124212,-0.012231,-0.053833,-0.088318,0.034581,0.044943,-0.036076,-0.031654,Count_Person_VisionDifficulty
456,0.111441,-0.040143,0.019932,0.023578,0.020069,0.079719,0.074978,0.065209,-0.016704,-0.002872,...,-0.080457,0.102178,-0.005626,-0.055989,-0.078098,0.022486,0.007207,-0.040646,-0.012215,Count_Person_VisionDifficulty
457,0.061499,-0.097763,-0.002136,0.012396,-0.030989,0.100217,0.051519,0.065312,-0.019083,0.015819,...,-0.015887,0.127537,-0.024968,-0.056477,-0.089059,0.022850,-0.003102,0.002228,-0.013532,Count_Person_VisionDifficulty
458,0.142949,-0.054331,0.040648,0.040143,-0.031601,0.103709,0.038664,0.073190,-0.042600,-0.000213,...,-0.052508,0.143047,-0.005861,-0.054197,-0.080286,0.017755,0.029367,-0.022975,-0.014500,Count_Person_VisionDifficulty


In [28]:
import torch
from datasets import load_dataset

# Mimic save / load of embeddings.  (Really only necessary when index becomes bigger)
embeddings.to_csv("embeddings_demographics300.csv", index=False)

TO_GCS = True
if TO_GCS:
  !gcloud config set project 'datcom-204919'
  !gsutil cp embeddings_demographics300.csv gs://datcom-csv/embeddings/

Updated property [core/project].
Copying file://embeddings_demographics300.csv [Content-Type=text/csv]...
\
Operation completed over 1 objects/2.1 MiB.                                      


In [30]:
# Load for Search below
ds = load_dataset('csv', data_files='embeddings_demographics300.csv')

df = ds["train"].to_pandas()
dcids = df['dcid'].values.tolist()
df = df.drop('dcid', axis=1)

dataset_embeddings = torch.from_numpy(df.to_numpy()).to(torch.float)

  0%|          | 0/1 [00:00<?, ?it/s]

## 3. Search

Given a search query, uses the model downloaded above (so run the above cells first!) to compute query embeddings, and calls the [`semantic_search`](https://www.sbert.net/examples/applications/semantic-search/README.html#util-semantic-search) function.  Per docs, the function computes exact nearest-neighbor using Cosine similarity match by default.

*Run the cell below once, and then auto-completion will work (sometimes needs a space at the end)*

TODO: Understand score values, especially for results unrelated to the query.

In [50]:
#@title { run: "auto", vertical-output: true }
QUERY = "poor people" #@param {type:"string"}

query_embeddings = model.encode([QUERY])

from sentence_transformers.util import semantic_search
hits = semantic_search(query_embeddings, dataset_embeddings, top_k=20)

# Note: multiple results may map to the same DCID. As well, the same string may
# map to multiple DCIDs with the same score.
sv2score = {}
score2svs = {}
for e in hits[0]:
  for d in dcids[e['corpus_id']].split(','):
    s = e['score']
    # Prefer the top score.
    if d not in sv2score:
      sv2score[d] = s
      if s not in score2svs:
        score2svs[s] = [d]
      else:
        score2svs[s].append(d)

scores = [s for s in sorted(score2svs.keys(), reverse=True)]
svs = [' : '.join(score2svs[s]) for s in scores]
result = pd.DataFrame({'SV': svs, 'Cosine Score': scores})

pd.set_option('max_colwidth', 400)

result

,SV,Cosine Score
0,Count_Person_BelowPovertyLevelInThePast12Months_SomeOtherRaceAlone,0.559105
1,Count_Person_AbovePovertyLevelInThePast12Months_SomeOtherRaceAlone,0.551792
2,Count_Person_BelowPovertyLevelInThePast12Months,0.538634
3,Count_Person_AbovePovertyLevelInThePast12Months,0.530051
4,Count_Person_BelowPovertyLevelInThePast12Months_TwoOrMoreRaces,0.529829
5,Count_Person_BelowPovertyLevelInThePast12Months_AsFractionOf_Count_Person,0.519758
6,Count_Person_AbovePovertyLevelInThePast12Months_TwoOrMoreRaces,0.517439
7,Count_Person_BelowPovertyLevelInThePast12Months_WhiteAlone,0.516282
8,Count_Person_PovertyStatusDetermined,0.516059
9,Count_Person_Urban_BelowPovertyLevelInThePast12Months,0.512605
